In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd

pattern_ecran = re.compile(r"\((.*?)\)")
pattern_conf_ecran = re.compile(r"<(.*?)>")
pattern_chaine = re.compile(r"\$(.*?)\$")

def read_ds(ds_name: str):
    with open(f"data/{ds_name}.csv") as f:
        max_actions = max((len(str(c).split(",")) for c in f.readlines()))
        f.seek(0)
        _names = ["util", "navigateur"] if "train" in ds_name else ["navigateur"]
        _names.extend(range(max_actions - len(_names)))
        return pd.read_csv(f, names=_names, dtype=str)


def to_categories(df, col="util"):
    df[[col]] = df[[col]].apply(lambda x: x.cat.codes)


def filter_action(value: str):
    for delim in ["(", "<", "$", "1"]:
        if delim in value and (low_ind := value.index(delim)):
            value = value[:low_ind]
    return value


def get_outliers(df):
    df_length = df.iloc[:, 2:].notna().sum(axis=1)
    log_length = np.log10(df_length)
    q1, q3 = log_length.quantile([0.25, 0.75])
    iqr = q3 - q1
    borne_basse, borne_haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = (log_length < borne_basse) | (log_length > borne_haute)

    # TODO : add removing duplicate times
    return outliers


def get_sessions(df):
    debut = 2 if "util" in df.columns else 1  # le test n'a pas de colonne util
    return [
        [v for v in ligne if isinstance(v, str)]
        for ligne in df.iloc[:, debut:].to_numpy()
    ]


def is_timestamp(a):
    return re.fullmatch(r"t\d+", a) is not None


def most_frequent(valeurs):
    # renvoie la valeur la plus fréquente de la liste, ou "aucun" si la liste est vide
    return Counter(valeurs).most_common(1)[0][0] if valeurs else "aucun"



In [2]:
features_train = read_ds("train")
features_test = read_ds("test")

features_train["util"] = pd.Categorical(features_train["util"])

uniques = features_train.iloc[:,2:].stack().dropna().unique()
filtered_uniques = list({filter_action(un) for un in uniques if not un.startswith("t")})

train_outliers = get_outliers(features_train)
cleaned_features_train = features_train[~train_outliers].reset_index(drop=True)
y = cleaned_features_train["util"].cat.codes.to_numpy()

sessions_train = get_sessions(cleaned_features_train)
sessions_test = get_sessions(features_test)


In [3]:
def build_features(sessions, filtered_uniques):
    lignes = []
    for s in sessions:
        actions = [a for a in s if not is_timestamp(a)]
        temps = [int(a[1:]) for a in s if is_timestamp(a)]
        n = max(len(actions), 1)
        
        action_counts = []
        current_count = 0
        for a in s:
            if is_timestamp(a):
                action_counts.append(current_count)
                current_count = 0
            else:
                current_count += 1
        # Capture any remaining actions after the last timestamp
        if current_count > 0 or not action_counts:
            action_counts.append(current_count)
            
        # Standard deviation of the action counts between timestamps
        actions_per_interval_std = np.std(action_counts) if len(action_counts) > 1 else 0
        f = {
            "n_actions": n,
            "n_actions_uniques": len(set(actions)),
            # "ratio_uniques": len(set(actions)) / n,
            "duree": max(temps, default=0),
            # "std": actions_per_interval_std
        }
        
        compte = Counter(filter_action(a) for a in actions)
        for type_action in filtered_uniques:
            f["freq_" + type_action] = compte.get(type_action, 0) / n
        lignes.append(f)

    return pd.DataFrame(lignes)

X_train_core = build_features(sessions_train, filtered_uniques).reset_index(drop=True)
X_test_core = build_features(sessions_test, filtered_uniques).reset_index(drop=True)

In [4]:
def regex_features(sessions):
    lignes = []
    for s in sessions:
        # Extract matches action by action
        ecrans = [m for a in s for m in pattern_ecran.findall(a)]
        confs = [m for a in s for m in pattern_conf_ecran.findall(a)]
        chaines = [m for a in s for m in pattern_chaine.findall(a)]
        
        # Calculate dominance ratio for the top screen (how focused the user is)
        ecran_counts = Counter(ecrans)
        top_ecran_ratio = ecran_counts.most_common(1)[0][1] / len(ecrans) if ecrans else 0.0
        
        conf_counts = Counter(confs)
        top_conf_ratio = conf_counts.most_common(1)[0][1] / len(confs) if confs else 0.0

        lignes.append({
            # Top elements
            "ecran_top": most_frequent(ecrans),
            "conf_top": most_frequent(confs),
            "chaine_top": most_frequent(chaines),
            
            # # Flow features (Entry & Exit points)
            # "ecran_first": ecrans[0] if ecrans else "aucun",
            # "ecran_last": ecrans[-1] if ecrans else "aucun",
            # "conf_first": confs[0] if confs else "aucun",
            # "conf_last": confs[-1] if confs else "aucun",
            
            # # Volume & Diversity features
            "n_ecrans_total": len(ecrans),
            # "n_ecrans_distincts": len(set(ecrans)),
            "n_confs_total": len(confs),
            # "n_confs_distinctes": len(set(confs)),
            "n_chaines_total": len(chaines),
            # "n_chaines_distinctes": len(set(chaines)),
            
            # # Behavioral focus ratio
            # "top_ecran_ratio": top_ecran_ratio,
            # "top_conf_ratio": top_conf_ratio,
        })
        
    return pd.DataFrame(lignes)

X_regex_train = regex_features(sessions_train).reset_index(drop=True)
X_regex_test = regex_features(sessions_test).reset_index(drop=True)

In [5]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

cat_features_train = pd.concat([
    cleaned_features_train[["navigateur"]], 
    X_regex_train
], axis=1)

cat_features_test = pd.concat([
    features_test[["navigateur"]], 
    X_regex_test
], axis=1)

categorical_cols = [
    "navigateur", 
    "ecran_top", "conf_top", "chaine_top", 
    # "ecran_first", "ecran_last", 
    # "conf_first", "conf_last"
]

for col in categorical_cols:
    if col in cat_features_train.columns:
        train_cat = pd.Categorical(cat_features_train[col])
        cat_features_train[col] = train_cat.codes
        
        cat_features_test[col] = pd.Categorical(
            cat_features_test[col], 
            categories=train_cat.categories
        ).codes

X_final_train = pd.concat([X_train_core, cat_features_train], axis=1)
X_final_test = pd.concat([X_test_core, cat_features_test], axis=1)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_final_train, y, test_size=0.2, random_state=42, stratify=y
)

/tmp/ipykernel_22204/3014914993.py:27: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  cat_features_test[col] = pd.Categorical(


In [7]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

from sklearn.ensemble import ExtraTreesClassifier

model = ExtraTreesClassifier(
    n_estimators=2000,
    random_state=42,
    # max_features="sqrt",
    n_jobs=-1
)

model.fit(X_tr, y_tr)
result = model.predict(X_val)
print(f"Validation Accuracy: {accuracy_score(y_val, result):.4f}")
print(f"F1 score: {f1_score(y_val, result, average="macro"):.4f}")

Validation Accuracy: 0.8840
F1 score: 0.8457


In [16]:
predictions_encoded = model.predict(X_final_test)
predictions = cleaned_features_train["util"].cat.categories[predictions_encoded]

submission = pd.DataFrame({
    'RowId': range(1, len(X_final_test) + 1), 
    'prediction': predictions
})

# 5. Save to a CSV file (index=False is crucial for Kaggle!)
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully!")

Submission file 'submission.csv' created successfully!
